# Matriz de transição aproximada do gráfico suavizado

In [ ]:
'''
from scipy.optimize import nnls

X_completo = np.array([S_normal_lowess, I_normal_lowess, R_normal_lowess, D_normal_lowess])
dia_pico = np.argmax(I_normal_lowess)
dia_pico+=30

X   = X_completo[:, :-1]
X_l = X_completo[:, 1:]

X_growth = X[:, :dia_pico]
X_l_growth = X_l[:, :dia_pico]

print('Matriz X:\n', X)
print("\nMatriz X':\n", X_l)

mascara = np.array([
    [True,  False, False, False],
    [True,  True,  False, False],
    [False, True,  True,  False],
    [False, True,  False, True ],
])

n_estados = X_growth.shape[0]
A = np.zeros((n_estados, n_estados))

for i in range(n_estados):
    colunas_permitidas = np.where(mascara[i])[0]
    a_reduzida = X_growth[colunas_permitidas, :].T
    b_linha = X_l_growth[i, :]

    coeficientes = nnls(a_reduzida, b_linha)[0]

    A[i, colunas_permitidas] = coeficientes

somas_colunas = A.sum(axis=0)
fator = np.maximum(somas_colunas, 1)
A = A/fator

print("\nMatriz A:\n", A)
'''

# Validando a matriz

In [ ]:
'''
dias = 94
x = X[:, 0]
historico = [x]

for t in range(dias):
    x = A @ x
    historico.append(x)
historico = np.array(historico)

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, len(historico)-1)
ax.set_yscale('log')
ax.set_ylim(1e-6, 1e-2)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (log)')
ax.set_title('Testando a matriz A')
ax.grid(True, linestyle='--', alpha=0.5)

cores = ['firebrick', 'mediumseagreen', 'dimgray'] 
labels = ['Infectados', 'Recuperados', 'Falecidos']
linhas = [ax.plot([], [], lw=2, color=cores[i], label=labels[i])[0] for i in range(3)]
ax.legend(loc='upper right')

def init():
    for i in range(3):
        linhas[i].set_data([], [])
    return linhas

def atualizar(frame):
    for i in range(3):
        linhas[i].set_data(list(range(frame+1)), historico[:frame+1, i+1])
    return linhas

animation = FuncAnimation(fig, atualizar, frames=len(historico), init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())
'''

> O gráfico gerado pela matriz **A** ainda não está fiel ao gráfico SIRD suavizado